In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv('dataset.csv')

In [3]:
"""
Task B — road closure classifier, CLEAN retrain
=================================================
Reuses Task A's exact feature-engineering pipeline and its already-fitted
encoders (label_encoders_clean.pkl, global_tenc_clean.pkl) so that
build_feature_row() in main.py can serve BOTH models from one feature dict,
with zero silent -999 fallbacks and zero risk of the two models disagreeing
on what an encoded value means.

Assumes `df` (the raw events dataframe) is already loaded in the environment,
same as the Task A script.

Run this AFTER the Task A clean script has been run at least once and has
saved models_v3/label_encoders_clean.pkl + models_v3/global_tenc_clean.pkl.
"""

import pandas as pd
import numpy as np
import joblib, os, warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

MODEL_DIR = 'models_v3'

# ─────────────────────────────────────────────────────────────────────────────
# 0. Load Task A's canonical encoders — TRANSFORM ONLY, never refit
# ─────────────────────────────────────────────────────────────────────────────
le_encoders = joblib.load(os.path.join(MODEL_DIR, 'label_encoders_clean.pkl'))
global_tenc = joblib.load(os.path.join(MODEL_DIR, 'global_tenc_clean.pkl'))

print("Loaded canonical encoders from Task A:")
for col, le in le_encoders.items():
    print(f"  {col}: {list(le.classes_)}")
print(f"  target-encoded columns: {list(global_tenc.keys())}")

# ─────────────────────────────────────────────────────────────────────────────
# 1. Datetime + duration  (identical to Task A)
# ─────────────────────────────────────────────────────────────────────────────
for col in ['closed_datetime', 'start_datetime', 'resolved_datetime', 'created_date']:
    df[col] = pd.to_datetime(df[col], utc=True, errors='coerce')

df['end_ts']        = df['closed_datetime'].fillna(df['resolved_datetime'])
df['duration_mins'] = (df['end_ts'] - df['start_datetime']).dt.total_seconds() / 60

mask = (
    df['duration_mins'].notna() &
    (df['duration_mins'] > 0) &
    (df['duration_mins'] < 1440) &
    df['priority'].notna() &
    df['address'].notna()
)
df_clean = df[mask].copy().reset_index(drop=True)
print(f"\nRows: {len(df_clean)}")

# ─────────────────────────────────────────────────────────────────────────────
# 2. Fill categoricals  (identical to Task A)
# ─────────────────────────────────────────────────────────────────────────────
for col in ['zone', 'junction', 'gba_identifier', 'veh_type',
            'corridor', 'event_cause', 'event_type']:
    df_clean[col] = df_clean[col].fillna('Unknown')

df_clean['requires_road_closure'] = df_clean['requires_road_closure'].astype(int)

# ─────────────────────────────────────────────────────────────────────────────
# 3. Temporal features  (identical to Task A)
# ─────────────────────────────────────────────────────────────────────────────
df_clean['hour']         = df_clean['start_datetime'].dt.hour
df_clean['day_of_week']  = df_clean['start_datetime'].dt.dayofweek
df_clean['month']        = df_clean['start_datetime'].dt.month
df_clean['week_of_year'] = df_clean['start_datetime'].dt.isocalendar().week.astype(int)
df_clean['quarter']      = df_clean['start_datetime'].dt.quarter
df_clean['is_weekend']   = df_clean['day_of_week'].isin([5, 6]).astype(int)
df_clean['is_peak']      = df_clean['hour'].isin([7, 8, 9, 17, 18, 19]).astype(int)
df_clean['is_peak_am']   = df_clean['hour'].isin([7, 8, 9]).astype(int)
df_clean['is_peak_pm']   = df_clean['hour'].isin([17, 18, 19]).astype(int)
df_clean['is_night']     = df_clean['hour'].isin([22, 23, 0, 1, 2]).astype(int)
df_clean['is_monday']    = (df_clean['day_of_week'] == 0).astype(int)
df_clean['is_friday']    = (df_clean['day_of_week'] == 4).astype(int)

df_clean['hour_sin']  = np.sin(2 * np.pi * df_clean['hour'] / 24)
df_clean['hour_cos']  = np.cos(2 * np.pi * df_clean['hour'] / 24)
df_clean['dow_sin']   = np.sin(2 * np.pi * df_clean['day_of_week'] / 7)
df_clean['dow_cos']   = np.cos(2 * np.pi * df_clean['day_of_week'] / 7)
df_clean['month_sin'] = np.sin(2 * np.pi * df_clean['month'] / 12)
df_clean['month_cos'] = np.cos(2 * np.pi * df_clean['month'] / 12)

# ─────────────────────────────────────────────────────────────────────────────
# 4. Geo features  (identical to Task A)
# ─────────────────────────────────────────────────────────────────────────────
df_clean['lat_bin']          = pd.cut(df_clean['latitude'], bins=8, labels=False)
df_clean['lon_bin']          = pd.cut(df_clean['longitude'], bins=8, labels=False)
df_clean['has_end_location'] = (df_clean['endlatitude'].fillna(0) != 0).astype(int)
df_clean['dist_from_center'] = np.sqrt(
    (df_clean['latitude'] - 12.9716) ** 2 + (df_clean['longitude'] - 77.5946) ** 2)

# ─────────────────────────────────────────────────────────────────────────────
# 5. Label encoding — TRANSFORM ONLY, using Task A's fitted encoders
# ─────────────────────────────────────────────────────────────────────────────
LE_COLS = ['event_type', 'event_cause', 'veh_type']

def safe_transform(le, series):
    """Map any category unseen by Task A's encoder to its first known class,
    same fallback logic main.py uses at inference (safe_le)."""
    known = set(le.classes_)
    vals  = series.astype(str).where(series.astype(str).isin(known), le.classes_[0])
    return le.transform(vals)

for col in LE_COLS:
    df_clean[f'{col}_enc'] = safe_transform(le_encoders[col], df_clean[col])

priority_map = {'low': 0, 'medium': 1, 'high': 2, 'critical': 3}
df_clean['priority_enc'] = df_clean['priority'].str.lower().map(
    priority_map).fillna(1).astype(int)

# ─────────────────────────────────────────────────────────────────────────────
# 6. Target encoding — MAP ONLY, using Task A's saved mean-duration maps
# ─────────────────────────────────────────────────────────────────────────────
TENC_COLS = ['zone', 'junction', 'corridor', 'event_cause']
global_duration_mean = df_clean['duration_mins'].mean()

for col in TENC_COLS:
    mean_map = global_tenc[col]
    df_clean[f'{col}_tenc']  = df_clean[col].map(mean_map).fillna(global_duration_mean)
    # NOTE: count_map is recomputed locally, same as Task A's training script
    # did — Task A never persisted count maps to disk either, so this is a
    # known, already-flagged limitation that's purely about INFERENCE-time
    # main.py (which falls back to 1.0). It doesn't affect training here.
    df_clean[f'{col}_count'] = df_clean[col].map(df_clean[col].value_counts()).fillna(0)

# ─────────────────────────────────────────────────────────────────────────────
# 7. Interaction features  (identical to Task A)
# ─────────────────────────────────────────────────────────────────────────────
df_clean['cause_x_peak']    = df_clean['event_cause_enc'] * df_clean['is_peak']
df_clean['cause_x_weekend'] = df_clean['event_cause_enc'] * df_clean['is_weekend']
df_clean['cause_x_closure'] = df_clean['event_cause_enc'] * df_clean['requires_road_closure']
df_clean['zone_x_peak']     = df_clean['zone_tenc'] * df_clean['is_peak']
df_clean['closure_x_end']   = df_clean['requires_road_closure'] * df_clean['has_end_location']

# ─────────────────────────────────────────────────────────────────────────────
# 8. FEATURE_COLS_A — Task A's list, reproduced here only so we can derive
#    FEATURE_COLS_B from it. Not used to train anything in this script.
# ─────────────────────────────────────────────────────────────────────────────
FEATURE_COLS_A = [
    'hour', 'day_of_week', 'month', 'week_of_year', 'quarter',
    'is_weekend', 'is_peak', 'is_peak_am', 'is_peak_pm',
    'is_night', 'is_monday', 'is_friday',
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos',
    'latitude', 'longitude', 'lat_bin', 'lon_bin',
    'has_end_location', 'dist_from_center',
    'event_type_enc', 'event_cause_enc', 'veh_type_enc',
    'requires_road_closure', 'priority_enc',
    'zone_tenc', 'zone_count',
    'junction_tenc', 'junction_count',
    'corridor_tenc', 'corridor_count',
    'event_cause_tenc', 'event_cause_count',
    'cause_x_peak', 'cause_x_weekend', 'cause_x_closure',
    'zone_x_peak', 'closure_x_end',
]
FEATURE_COLS_A = [c for c in FEATURE_COLS_A if c in df_clean.columns]

# ─────────────────────────────────────────────────────────────────────────────
# 9. FEATURE_COLS_B — Task A's list minus the target and its leaky derivatives
#
#    requires_road_closure  → IS the target. Never a feature.
#    has_end_location       → near-perfect closure tell. Same exclusion the
#                              original Task B script made, for the same
#                              reason (it would make the model lazy/leaky).
#    cause_x_closure         → literally event_cause_enc * requires_road_closure
#    closure_x_end           → literally requires_road_closure * has_end_location
#    The two interaction terms multiply the TARGET directly into a feature —
#    keeping either would leak the label almost verbatim. They only belong
#    in Task A (duration regression), where requires_road_closure is a valid
#    *input* rather than the thing being predicted.
# ─────────────────────────────────────────────────────────────────────────────
LEAKY_FOR_B = {'requires_road_closure', 'has_end_location', 'cause_x_closure', 'closure_x_end'}
FEATURE_COLS_B = [c for c in FEATURE_COLS_A if c not in LEAKY_FOR_B]

print(f"\nFEATURE_COLS_A: {len(FEATURE_COLS_A)} features")
print(f"FEATURE_COLS_B: {len(FEATURE_COLS_B)} features")
print(f"Dropped for Task B (leakage): {sorted(LEAKY_FOR_B & set(FEATURE_COLS_A))}")

# Quick leakage sanity check — these correlations should look suspiciously high
print("\nCorrelation with target (requires_road_closure) — sanity check:")
for c in ['has_end_location', 'cause_x_closure', 'closure_x_end']:
    if c in df_clean.columns:
        corr = df_clean[c].astype(float).corr(df_clean['requires_road_closure'].astype(float))
        print(f"  {c:20s} corr = {corr:.4f}  (excluded from FEATURE_COLS_B)")

# ─────────────────────────────────────────────────────────────────────────────
# 10. Build X / y for Task B
# ─────────────────────────────────────────────────────────────────────────────
X_b = df_clean[FEATURE_COLS_B].fillna(-999)
y_b = df_clean['requires_road_closure'].astype(int)

print("\nClass balance — requires_road_closure:")
print(y_b.value_counts(normalize=True).rename('proportion').to_string())

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ─────────────────────────────────────────────────────────────────────────────
# 11. Optuna — tune XGBClassifier on OOF AUC
# ─────────────────────────────────────────────────────────────────────────────
print("\nRunning Optuna (60 trials)...")

neg, pos = (y_b == 0).sum(), (y_b == 1).sum()
spw_global = neg / pos

def objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 200, 800),
        'max_depth':        trial.suggest_int('max_depth', 3, 7),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma':            trial.suggest_float('gamma', 0.0, 1.0),
        'reg_alpha':        trial.suggest_float('reg_alpha', 0.0, 3.0),
        'reg_lambda':       trial.suggest_float('reg_lambda', 0.5, 5.0),
        'scale_pos_weight': spw_global,
        'eval_metric':      'aucpr',
        'random_state': 42, 'verbosity': 0, 'n_jobs': -1,
    }
    aucs = []
    for tr_idx, va_idx in skf.split(X_b, y_b):
        m = xgb.XGBClassifier(**params)
        m.fit(X_b.iloc[tr_idx], y_b.iloc[tr_idx])
        prob = m.predict_proba(X_b.iloc[va_idx])[:, 1]
        aucs.append(roc_auc_score(y_b.iloc[va_idx], prob))
    return np.mean(aucs)

study = optuna.create_study(direction='maximize',
                             sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=60, show_progress_bar=True)
best_params = dict(study.best_params)
best_params.update({
    'scale_pos_weight': spw_global, 'eval_metric': 'aucpr',
    'random_state': 42, 'verbosity': 0, 'n_jobs': -1,
})
print(f"\nBest OOF AUC: {study.best_value:.4f}")

# ─────────────────────────────────────────────────────────────────────────────
# 12. OOF evaluation — XGBoost vs Logistic Regression vs Random Forest
# ─────────────────────────────────────────────────────────────────────────────
oof_prob_xgb = np.zeros(len(X_b))
for tr_idx, va_idx in skf.split(X_b, y_b):
    m = xgb.XGBClassifier(**best_params)
    m.fit(X_b.iloc[tr_idx], y_b.iloc[tr_idx])
    oof_prob_xgb[va_idx] = m.predict_proba(X_b.iloc[va_idx])[:, 1]
oof_pred_xgb = (oof_prob_xgb >= 0.5).astype(int)

results = [{
    'model': 'XGBoost',
    'F1':  f1_score(y_b, oof_pred_xgb, average='weighted'),
    'AUC': roc_auc_score(y_b, oof_prob_xgb),
}]

lr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(class_weight='balanced', max_iter=500, random_state=42)),
])
oof_prob_lr = np.zeros(len(X_b))
for tr_idx, va_idx in skf.split(X_b, y_b):
    lr_pipe.fit(X_b.iloc[tr_idx], y_b.iloc[tr_idx])
    oof_prob_lr[va_idx] = lr_pipe.predict_proba(X_b.iloc[va_idx])[:, 1]
oof_pred_lr = (oof_prob_lr >= 0.5).astype(int)
results.append({
    'model': 'LogisticReg',
    'F1':  f1_score(y_b, oof_pred_lr, average='weighted'),
    'AUC': roc_auc_score(y_b, oof_prob_lr),
})

rf_clf = RandomForestClassifier(
    n_estimators=200, max_depth=8, class_weight='balanced',
    random_state=42, n_jobs=-1,
)
oof_prob_rf = np.zeros(len(X_b))
for tr_idx, va_idx in skf.split(X_b, y_b):
    rf_clf.fit(X_b.iloc[tr_idx], y_b.iloc[tr_idx])
    oof_prob_rf[va_idx] = rf_clf.predict_proba(X_b.iloc[va_idx])[:, 1]
oof_pred_rf = (oof_prob_rf >= 0.5).astype(int)
results.append({
    'model': 'RandomForest',
    'F1':  f1_score(y_b, oof_pred_rf, average='weighted'),
    'AUC': roc_auc_score(y_b, oof_prob_rf),
})

print(f"\n{'='*55}")
print("  TASK B — CLEAN MODEL RESULTS (OOF, no leakage)")
print(pd.DataFrame(results).sort_values('AUC', ascending=False).to_string(index=False))
print(f"{'='*55}")

print("\nXGBoost OOF classification report:")
print(classification_report(y_b, oof_pred_xgb, target_names=['No closure', 'Closure']))

# ─────────────────────────────────────────────────────────────────────────────
# 13. Fit final model on all data
# ─────────────────────────────────────────────────────────────────────────────
final_model_b = xgb.XGBClassifier(**best_params)
final_model_b.fit(X_b, y_b)

# ─────────────────────────────────────────────────────────────────────────────
# 14. Sanity check predictions
# ─────────────────────────────────────────────────────────────────────────────
print("\nSanity checks on trained Task B model:")
feat_idx_b     = {f: i for i, f in enumerate(FEATURE_COLS_B)}
global_means_b = {col: float(df_clean[col].mean()) for col in FEATURE_COLS_B}

def make_test_b(overrides):
    row = np.array([global_means_b[f] for f in FEATURE_COLS_B], dtype=np.float32)
    for k, v in overrides.items():
        if k in feat_idx_b:
            row[feat_idx_b[k]] = v
    return row.reshape(1, -1)

cause_classes = list(le_encoders['event_cause'].classes_)

cases_b = [
    ("Minor breakdown, off-peak — closure unlikely", {
        'event_cause_enc':  cause_classes.index('vehicle_breakdown'),
        'event_cause_tenc': global_tenc['event_cause'].get('vehicle_breakdown', 50),
        'hour': 14, 'is_peak': 0, 'is_peak_am': 0, 'is_peak_pm': 0,
        'cause_x_peak': 0, 'cause_x_weekend': 0,
    }),
    ("Tree fall, morning peak — closure likely", {
        'event_cause_enc':  cause_classes.index('tree_fall'),
        'event_cause_tenc': global_tenc['event_cause'].get('tree_fall', 89),
        'hour': 8, 'is_peak': 1, 'is_peak_am': 1, 'is_peak_pm': 0,
        'zone_tenc': global_tenc['zone'].get('North Zone 1', 89),
    }),
    ("Procession, peak hour, central zone — closure very likely", {
        'event_cause_enc':  cause_classes.index('procession'),
        'event_cause_tenc': global_tenc['event_cause'].get('procession', 89),
        'hour': 8, 'is_peak': 1, 'is_peak_am': 1, 'is_peak_pm': 0,
        'zone_tenc': global_tenc['zone'].get('Central Zone 1', 89),
        'cause_x_peak': cause_classes.index('procession'),
    }),
]

for name, overrides in cases_b:
    X_test = make_test_b(overrides)
    prob   = final_model_b.predict_proba(X_test)[0][1]
    print(f"  {name:60s} → {prob:.3f} closure probability")

# ─────────────────────────────────────────────────────────────────────────────
# 15. Save — only NEW artifacts. Encoders / tenc maps are reused from Task A
#     as-is and must NOT be re-saved here (this script never refit them).
# ─────────────────────────────────────────────────────────────────────────────
os.makedirs(MODEL_DIR, exist_ok=True)
joblib.dump(final_model_b,  os.path.join(MODEL_DIR, 'xgb_road_closure.pkl'))
joblib.dump(FEATURE_COLS_B, os.path.join(MODEL_DIR, 'feature_cols_B.pkl'))
joblib.dump(best_params,    os.path.join(MODEL_DIR, 'best_params_B.pkl'))

print(f"\nSaved to {MODEL_DIR}/")
print("  xgb_road_closure.pkl  ← copy to backend/models/, replaces the old one")
print("  feature_cols_B.pkl    ← copy to backend/models/, replaces the old one")
print("  label_encoders_clean.pkl and global_tenc_clean.pkl are UNCHANGED —")
print("  this script reused Task A's copies directly. Do not overwrite them")
print("  with anything refit here; there's nothing new to copy for those two.")

# ─────────────────────────────────────────────────────────────────────────────
# 16. Feature importances
# ─────────────────────────────────────────────────────────────────────────────
importances_b = final_model_b.feature_importances_
imp_pairs_b = sorted(zip(FEATURE_COLS_B, importances_b), key=lambda x: x[1], reverse=True)
print("\nTop 10 feature importances (Task B clean model):")
for name, imp in imp_pairs_b[:10]:
    print(f"  {name:30s}  {imp:.4f}")

Loaded canonical encoders from Task A:
  event_type: ['planned', 'unplanned']
  event_cause: ['accident', 'congestion', 'construction', 'others', 'pot_holes', 'procession', 'protest', 'road_conditions', 'test_demo', 'tree_fall', 'vehicle_breakdown', 'water_logging']
  veh_type: ['Unknown', 'auto', 'bmtc_bus', 'heavy_vehicle', 'ksrtc_bus', 'lcv', 'others', 'private_bus', 'private_car', 'taxi', 'truck']
  target-encoded columns: ['zone', 'junction', 'corridor', 'event_cause']

Rows: 2523

FEATURE_COLS_A: 42 features
FEATURE_COLS_B: 38 features
Dropped for Task B (leakage): ['cause_x_closure', 'closure_x_end', 'has_end_location', 'requires_road_closure']

Correlation with target (requires_road_closure) — sanity check:
  has_end_location     corr = 0.9887  (excluded from FEATURE_COLS_B)
  cause_x_closure      corr = 0.9242  (excluded from FEATURE_COLS_B)
  closure_x_end        corr = 1.0000  (excluded from FEATURE_COLS_B)

Class balance — requires_road_closure:
requires_road_closure
0    0

  0%|          | 0/60 [00:00<?, ?it/s]


Best OOF AUC: 0.7440

  TASK B — CLEAN MODEL RESULTS (OOF, no leakage)
       model       F1      AUC
     XGBoost 0.825205 0.741885
RandomForest 0.907684 0.719255
 LogisticReg 0.780154 0.710939

XGBoost OOF classification report:
              precision    recall  f1-score   support

  No closure       0.96      0.79      0.87      2335
     Closure       0.19      0.58      0.28       188

    accuracy                           0.78      2523
   macro avg       0.57      0.69      0.57      2523
weighted avg       0.90      0.78      0.83      2523


Sanity checks on trained Task B model:
  Minor breakdown, off-peak — closure unlikely                 → 0.428 closure probability
  Tree fall, morning peak — closure likely                     → 0.730 closure probability
  Procession, peak hour, central zone — closure very likely    → 0.558 closure probability

Saved to models_v3/
  xgb_road_closure.pkl  ← copy to backend/models/, replaces the old one
  feature_cols_B.pkl    ← copy to b